# Tutorial: GovernAI Multi-Agent Workflow (Bounded Handoff)

This notebook runs a governed multi-agent workflow where:
- `research_agent` can only use allowlisted research tools
- it can only hand off to allowed agents
- runtime enforces deterministic transition + audit trail


## Outline

- Load API keys from adjacent `letstravel/.env`
- Define LLM-backed tools
- Define bounded agents with allowlisted tools/handoffs
- Run workflow and inspect handoff audit events


In [ ]:
import asyncio
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

REPO_ROOT = Path.cwd().resolve().parent.parent if "notebooks" in str(Path.cwd()) else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')
print('Loaded:', REPO_ROOT / '.env')
print('OPENAI_API_KEY present:', bool(os.getenv('OPENAI_API_KEY')))
print('OPENAI model:', os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))


## Step 1 - Define LLM Tools, Agents, and Workflow


In [ ]:
from pydantic import BaseModel

from governai import Agent, AgentResult, EventType, ToolRegistry, Workflow, step, tool


def make_llm() -> ChatOpenAI:
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('Missing OPENAI_API_KEY in local .env')
    return ChatOpenAI(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        api_key=api_key,
        temperature=0.2,
    )


class UserQuestion(BaseModel):
    question: str


class ResearchPayload(BaseModel):
    question: str


class ResearchResult(BaseModel):
    question: str
    research: str


class DraftResult(BaseModel):
    answer: str


@tool(name='llm.research', input_model=ResearchPayload, output_model=ResearchResult, capabilities=['llm.invoke'])
async def llm_research(ctx, data: ResearchPayload) -> ResearchResult:
    llm = make_llm()
    result = llm.invoke([
        SystemMessage(content='You are a precise research assistant. Keep output concise and factual.'),
        HumanMessage(content=f"Question: {data.question}\nProvide 5 concise factual research bullets."),
    ])
    return ResearchResult(question=data.question, research=result.content)


@tool(name='llm.draft', input_model=ResearchResult, output_model=DraftResult, capabilities=['llm.invoke'])
async def llm_draft(ctx, data: ResearchResult) -> DraftResult:
    llm = make_llm()
    result = llm.invoke([
        SystemMessage(content='You write clear final responses for users.'),
        HumanMessage(content=(
            f"Question: {data.question}\n"
            f"Research notes:\n{data.research}\n\n"
            'Write a final answer in 6-10 bullet points.'
        )),
    ])
    return DraftResult(answer=result.content)


async def research_agent_handler(ctx, task):
    research_out = await ctx.use_tool('llm.research', {'question': task.input_payload['question']})
    return AgentResult(
        status='handoff',
        next_agent='draft_agent',
        output_payload=research_out,
    )


async def draft_agent_handler(ctx, task):
    draft_out = await ctx.use_tool('llm.draft', task.input_payload)
    return AgentResult(status='final', output_payload=draft_out)


research_agent = Agent(
    name='research_agent',
    description='Researches the question using allowlisted tools',
    instruction='Gather factual research then hand off to drafting agent.',
    handler=research_agent_handler,
    input_model=UserQuestion,
    output_model=ResearchResult,
    allowed_tools=['llm.research'],
    allowed_handoffs=['draft_agent'],
    max_turns=1,
    max_tool_calls=1,
)

draft_agent = Agent(
    name='draft_agent',
    description='Writes final answer from research notes',
    instruction='Draft final user-facing answer.',
    handler=draft_agent_handler,
    input_model=ResearchResult,
    output_model=DraftResult,
    allowed_tools=['llm.draft'],
    allowed_handoffs=[],
    max_turns=1,
    max_tool_calls=1,
)


class MultiAgentResearchFlow(Workflow[UserQuestion, DraftResult]):
    research = step('research', agent=research_agent).then('draft')
    draft = step('draft', agent=draft_agent).then_end()


## Step 2 - Run and Inspect Handoff Governance


In [ ]:
tool_registry = ToolRegistry()
tool_registry.register(llm_research)
tool_registry.register(llm_draft)


async def run_multi_agent(question: str):
    flow = MultiAgentResearchFlow(tool_registry=tool_registry)
    state = await flow.run(UserQuestion(question=question))

    print('Status:', state.status)
    print('Completed steps:', state.completed_steps)
    print('\nFinal answer preview:\n')
    print(state.artifacts['draft']['answer'][:1000])

    print('\nHandoff-related audit events:')
    for event in flow.runtime.audit_emitter.events:
        if event.event_type in {
            EventType.AGENT_HANDOFF_PROPOSED,
            EventType.AGENT_HANDOFF_ACCEPTED,
            EventType.AGENT_HANDOFF_REJECTED,
            EventType.AGENT_ENTERED,
            EventType.AGENT_COMPLETED,
        }:
            print(f"- {event.event_type.value:28s} step={event.step_name} payload={event.payload}")

    return state


multi_state = asyncio.run(run_multi_agent('How should a startup choose between RAG and fine-tuning for support bots?'))
